# 1. Install Required Libraries

This section installs the Python libraries needed for the notebook. These libraries provide the functionality required to:

- Read environment variables and API keys securely.
- Send requests to the OpenAlex API.
- Organize and manipulate the retrieved data in a structured table.

Installing these packages ensures that all required dependencies are available before the rest of the notebook is executed.

In [9]:
#%pip install python-dotenv requests pandas


[notice] A new release of pip is available: 24.3.1 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# 2. Configure the OpenAlex API Connection

This section sets up the connection to the OpenAlex API.

It loads the OpenAlex API key (if available) from a secure environment file and specifies an email address that OpenAlex uses for identification. Providing an email address allows OpenAlex to place requests in its "polite pool," which generally results in more reliable and efficient access.

The base URL of the OpenAlex Works API is also stored for use in subsequent requests.


In [10]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

OPENALEX_API_KEY = os.getenv("https://api.openalex.org/works")
MAILTO = "shahs@lawrence.edu"

BASE_URL = "https://api.openalex.org/works"

# 3. Define the Search Parameters

This section defines the search criteria used to retrieve academic literature.

The search query includes multiple terms commonly associated with chatbots and conversational artificial intelligence, such as:

- Chatbot
- Conversational AI
- Virtual Assistant
- Dialogue System

Using multiple related keywords increases the likelihood of capturing a broad range of relevant publications.

Two additional parameters are specified:

- **Minimum Articles:** At least 50 articles should be collected.
- **Batch Size:** Up to 100 articles are requested per API call, which is the maximum number of records returned in a single response.


In [11]:
SEARCH_B = '''
("chatbot" OR "chatbots"
OR "chatterbot" OR "chatterbots"
OR "conversational agent" OR "conversational agents"
OR "embodied conversational agent" OR "embodied conversational agents"
OR "dialogue system" OR "dialogue systems"
OR "dialog system" OR "dialog systems"
OR "spoken dialogue system" OR "spoken dialogue systems"
OR "virtual assistant" OR "virtual assistants"
OR "voice assistant" OR "voice assistants"
OR "conversational AI"
OR "natural language interface" OR "natural language interfaces")
'''

MIN_RESULTS = 50
PER_PAGE = 100

# 4. Create a Helper Function to Reconstruct Abstracts

OpenAlex stores article abstracts in an **inverted index** format rather than as plain text. Instead of returning complete paragraphs, it provides each word along with its position in the abstract.

This section defines a helper function that reconstructs these inverted indexes into readable paragraphs. By converting the abstract back into normal text, the resulting dataset becomes much easier to analyze and interpret.

In [12]:
def reconstruct_abstract(inverted_index):
    if not inverted_index:
        return ""
    positions = {}
    for word, idxs in inverted_index.items():
        for idx in idxs:
            positions[idx] = word
    return " ".join(positions[i] for i in sorted(positions))

# 5. Retrieve Articles from OpenAlex

This section contains the main search function responsible for retrieving publications from OpenAlex.

The function sends the predefined search query to the API and retrieves matching articles in batches. Since each API request is limited to a maximum of 100 records, pagination is used to request additional batches until the required number of articles has been collected.

This automated process ensures that at least 50 relevant publications are retrieved efficiently.


In [13]:
def search_openalex_works(search_query, min_results=50, per_page=100, api_key=None, mailto=None):
    results = []
    cursor = "*"
    total_count = None

    params = {
        "search.title_and_abstract.exact": search_query,
        "per-page": per_page,
    }
    if mailto:
        params["mailto"] = mailto
    if api_key:
        params["api_key"] = api_key

    while True:
        params["cursor"] = cursor
        resp = requests.get(BASE_URL, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()

        if total_count is None:
            total_count = payload["meta"]["count"]
            print(f"OpenAlex reports {total_count} total matching works.")

        results.extend(payload["results"])
        cursor = payload["meta"].get("next_cursor")

        print(f"  fetched {len(results)} / {total_count} so far...")

        if not cursor or len(results) >= min_results or len(results) >= total_count:
            break

        time.sleep(0.2)

    return results, total_count

# 6. Execute the Search

This section runs the search function and begins retrieving articles from OpenAlex.

After the search is complete, the notebook displays:

- The total number of matching articles available in the OpenAlex database.
- The number of articles successfully retrieved.

As a quality check, the notebook verifies that at least 50 articles were collected. If fewer than the required number are returned, execution stops and displays an appropriate warning.


In [14]:
raw_results, total_available = search_openalex_works(
    SEARCH_B,
    min_results=MIN_RESULTS,
    per_page=PER_PAGE,
    api_key=OPENALEX_API_KEY,
    mailto=MAILTO,
)

print(f"\nCollected {len(raw_results)} works (of {total_available} total matches).")
assert len(raw_results) >= MIN_RESULTS, "Fewer than 50 results were returned — try broadening SEARCH_B."

OpenAlex reports 102381 total matching works.
  fetched 100 / 102381 so far...

Collected 100 works (of 102381 total matches).


# 7. Organize the Retrieved Data

The raw API response contains a large amount of metadata for each publication. This section extracts the most relevant information and organizes it into a structured table.

The following information is collected for each article:

- Article title
- Reconstructed abstract
- Author(s)
- Publication year
- Journal or publication venue
- DOI (Digital Object Identifier)
- Citation count
- Open access availability
- URL to the publication

The extracted information is then stored in a Pandas DataFrame, creating a clean, spreadsheet like dataset that is suitable for further analysis.


In [15]:
def parse_work(work):
    authors = [
        a["author"]["display_name"]
        for a in work.get("authorships", [])
        if a.get("author")
    ]
    primary_location = work.get("primary_location") or {}
    source = primary_location.get("source") or {}

    return {
        "openalex_id": work.get("id"),
        "title": work.get("title"),
        "abstract": reconstruct_abstract(work.get("abstract_inverted_index")),
        "publication_year": work.get("publication_year"),
        "publication_date": work.get("publication_date"),
        "authors": "; ".join(authors),
        "venue": source.get("display_name"),
        "doi": work.get("doi"),
        "cited_by_count": work.get("cited_by_count"),
        "is_oa": (work.get("open_access") or {}).get("is_oa"),
        "landing_page_url": primary_location.get("landing_page_url"),
    }

rows = [parse_work(w) for w in raw_results]
df = pd.DataFrame(rows)
df.head()

,openalex_id,title,abstract,publication_year,publication_date,authors,venue,doi,cited_by_count,is_oa,landing_page_url
0,https://openalex.org/W216945601,Embodied Conversational Agents,This book describes research in all aspects of...,2000,2000-03-27,,The MIT Press eBooks,https://doi.org/10.7551/mitpress/2697.001.0001,1792,False,https://doi.org/10.7551/mitpress/2697.001.0001
1,https://openalex.org/W1948566616,Semantically Conditioned LSTM-based Natural La...,Natural language generation (NLG) is a critica...,2015,2015-01-01,Tsung-Hsien Wen; Milica Gašić; Nikola Mrkšić; ...,None,https://doi.org/10.18653/v1/d15-1199,846,True,https://doi.org/10.18653/v1/d15-1199
2,https://openalex.org/W2612630960,Embodied Conversational Agents in Clinical Psy...,BACKGROUND: Embodied conversational agents (EC...,2017,2017-05-09,Simon Provoost; Ho Ming Lau; Jeroen Ruwaard; H...,Journal of Medical Internet Research,https://doi.org/10.2196/jmir.6553,450,True,https://doi.org/10.2196/jmir.6553
3,https://openalex.org/W1555414560,Social Dialongue with Embodied Conversational ...,,2005,2005-01-01,Timothy Bickmore; Justine Cassell,"Text, speech and language technology",https://doi.org/10.1007/1-4020-3933-6_2,321,False,https://doi.org/10.1007/1-4020-3933-6_2
4,https://openalex.org/W1517870057,Embodied conversational agents: representation...,How do we decide how to represent an intellige...,2001,2001-10-01,Justine Cassell,AI Magazine,https://doi.org/10.1609/aimag.v22i4.1593,270,False,https://dblp.uni-trier.de/db/journals/aim/aim2...


# 8. Clean and Export the Dataset

The final step prepares the dataset for future use.

First, duplicate records are removed to ensure that each publication appears only once in the final dataset. The cleaned data is then exported as a CSV file named:

**`openalex_search_b_results.csv`**

This CSV file can be opened in Microsoft Excel, Google Sheets, or any spreadsheet software, providing a permanent copy of the retrieved publications and their associated metadata.


In [16]:
df = df.drop_duplicates(subset="openalex_id").reset_index(drop=True)
print(f"Final dataset: {len(df)} unique articles")

output_path = "openalex_search_b_results.csv"
df.to_csv(output_path, index=False)
print(f"Saved to {output_path}")

Final dataset: 100 unique articles
Saved to openalex_search_b_results.csv


# Final Output

Upon completion of the notebook, a CSV dataset containing **100 academic publications** related to chatbots and conversational artificial intelligence is generated.

The dataset includes publication titles, reconstructed abstracts, authors, publication details, citation information, open-access status, and article links. This structured dataset can be used for subsequent analyses, such as text mining, sentiment analysis, topic modeling, or other natural language processing tasks.